<h1><font color="#113D68" size=5>Deep Learning para Procesamiento del Lenguaje Natural</font></h1>



<h1><font color="#113D68" size=6>2. Procesamiento de texto con scikit-learn</font></h1>



---

<a id="indice"></a>
<h2><font color="#004D7F" size=5>Índice</font></h2>

* [0. Contexto](#section0)
* [1. El modelo de la bolsa de palabras](#section1)
* [2. Contar palabras con `CountVectorizer`](#section2)
* [3. Frecuencias de palabras con `TfidfVectorizer`](#section3)
* [4. Hashing con `HashingVectorizer`](#section4)

---
<a id="section0"></a>
# <font color="#004D7F" size=6> 0. Contexto</font>

El texto debe analizarse para eliminar palabras (__tokenización__). Luego, las palabras deben codificarse como números enteros o valores de punto flotante para usar como entrada en un algoritmo de aprendizaje automático, i.e., extracción de características (__vectorización__). En este tutorial, aprenderemos como procesar texto con scikit-learn, específicamente veremos:
- Convertir texto en vectores de conteo de palabras con `CountVectorizer`.
- Convertir texto a vectores de frecuencia de palabras con `TfidfVectorizer`.
- Convertir texto en enteros únicos con `HashingVectorizer`.

<a id="section1"></a>
# <font color="#004D7F" size=6>1. El modelo de la bolsa de palabras</font>

No se puede trabajar con texto directamente cuando usamos algoritmos de Machine Learning, sino que debemos convertir el texto a números. Los algoritmos toman vectores de números como entrada, por lo tanto, necesitamos convertir el documento en vectores de números de longitud fija.

El modelo de bolsa de palabras (__BoW__, por sus siglas del inglés) elimina toda la información de orden en las palabras y se enfoca en la aparición de palabras en un documento, asignando a cada palabra un número único. Entonces:
- Cualquier documento que veamos puede codificarse como un vector de longitud fija con la longitud del vocabulario de palabras conocidas. 
- El valor en cada posición en el vector podría completarse con un recuento o frecuencia de cada palabra en el documento codificado.



La biblioteca scikit-learn proporciona 3 esquemas diferentes que podemos usar.

<a id="section2"></a>
# <font color="#004D7F" size=6>2. Contar de palabras con `CountVectorizer`</font>

`CountVectorizer` proporciona:
- Una forma sencilla de tokenizar una colección de documentos de texto y crear un vocabulario de palabras conocidas.
- Codificar nuevos documentos utilizando ese vocabulario. 

El procedimiento de uso:
1. Crear una instancia de la clase `CountVectorizer`.
2. Llamar a la función `fit()` para aprender un vocabulario de uno o más documentos.
3. Llamar a la función `transform()` en uno o más documentos según sea necesario para codificar cada uno como un vector.
4. Se devuelve un vector codificado con la longitud de todo el vocabulario y un número entero para el número de veces que apareció cada palabra en el documento. 
    - Debido a que estos vectores contendrán muchos ceros, los llamamos dispersos. 
    - El paquete `scipy.sparse` proporciona una manera eficiente de manejar vectores dispersos. 
5. Los vectores devueltos por una llamada a `transform()` serán vectores dispersos
6. Puede volver a transformarlos en matrices NumPy con la función `toarray()`. 

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Más información sobre la clase [`CountVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html)

In [53]:
from sklearn.feature_extraction.text import CountVectorizer

# 1. Definición de un corpus de texto de prueba (Muestra en español).
# Creamos una lista de documentos cortos para observar cómo el vectorizador mapea la frecuencia.
# Incluimos repeticiones intencionales de palabras clave para auditar el conteo.
corpus = [
    'Gregor Samsa despertó una mañana de un sueño tranquilo.',
    'El sueño de Gregor no era un sueño normal.',
    'Una mañana tranquila y una tarde tranquilo.'
]

# 2. Instanciación del modelo CountVectorizer.
# Este componente de scikit-learn convierte una colección de documentos de texto en una matriz de conteos de tokens.
# - Por defecto, aplica su propio tokenizador interno y convierte todo el texto a minúsculas (lowercase=True).
vectorizador = CountVectorizer()

# 3. Ajuste del modelo sobre el conjunto de entrenamiento (Fit).
# El método .fit() recorre todo el 'corpus' con tres objetivos principales:
#   a) Tokenizar el texto de manera interna utilizando sus propios patrones de expresiones regulares.
#   b) Normalizar los tokens encontrados (por defecto, los convierte completamente a minúsculas).
#   c) Construir el vocabulario único, donde mapea y asigna un índice numérico fijo a cada palabra descubierta.
# Nota técnica: En este punto el modelo ya conoce la estructura y dimensiones del vocabulario, 
# pero aún NO ha generado la matriz de frecuencias (no ha vectorizado los documentos).
vectorizador.fit(corpus)

CountVectorizer()

In [55]:
print(vectorizador.vocabulary_)

{'gregor': 4, 'samsa': 8, 'despertó': 1, 'una': 14, 'mañana': 5, 'de': 0, 'un': 13, 'sueño': 9, 'tranquilo': 12, 'el': 2, 'no': 6, 'era': 3, 'normal': 7, 'tranquila': 11, 'tarde': 10}


In [57]:
# 4. Transformación de los documentos en vectores de frecuencias (Transform).
# Utiliza el vocabulario previamente aprendido en el método .fit() para contar las
# apariciones de cada palabra en cada documento del corpus.
# Nota técnica: Genera una matriz dispersa (Compressed Sparse Row matrix) de SciPy. Esto es una
# optimización crucial en memoria para NLP, ya que la mayoría de las posiciones en textos reales serán 0.
matriz_frecuencias = vectorizador.transform(corpus)

# 5. Extracción de las características del vocabulario.
# get_feature_names_out() recupera los tokens únicos que conforman el orden de las columnas de la matriz.
# Los nombres están ordenados alfabéticamente de forma automática.
vocabulario = vectorizador.get_feature_names_out()

# 6. Inspección de resultados en Colab de forma legible.
# Pasamos la matriz dispersa a un arreglo denso (.toarray()) para visualizar la matriz de Bag of Words.
print("=== VOCABULARIO APRENDIDO (COLUMNAS) ===")
print(f""" 
Vectorizador: {vectorizador.vocabulary_}

Lista: {list(vocabulario)}

Tamaño de vocabulario: {len(vocabulario)}

Corpus: {corpus}
""")
print("\n" + "="*60 + "\n")

print("=== MATRIZ DE CONTEO EN FORMATO DENSO ===")
# Cada fila mapea el documento correspondiente del corpus en función de los índices del vocabulario.
print(matriz_frecuencias.toarray())

=== VOCABULARIO APRENDIDO (COLUMNAS) ===
 
Vectorizador: {'gregor': 4, 'samsa': 8, 'despertó': 1, 'una': 14, 'mañana': 5, 'de': 0, 'un': 13, 'sueño': 9, 'tranquilo': 12, 'el': 2, 'no': 6, 'era': 3, 'normal': 7, 'tranquila': 11, 'tarde': 10}

Lista: ['de', 'despertó', 'el', 'era', 'gregor', 'mañana', 'no', 'normal', 'samsa', 'sueño', 'tarde', 'tranquila', 'tranquilo', 'un', 'una']

Tamaño de vocabulario: 15

Corpus: ['Gregor Samsa despertó una mañana de un sueño tranquilo.', 'El sueño de Gregor no era un sueño normal.', 'Una mañana tranquila y una tarde tranquilo.']



=== MATRIZ DE CONTEO EN FORMATO DENSO ===
[[1 1 0 0 1 1 0 0 1 1 0 0 1 1 1]
 [1 0 1 1 1 0 1 1 0 2 0 0 0 1 0]
 [0 0 0 0 0 1 0 0 0 0 1 1 1 0 2]]


- **Estructura del Vocabulario (Columnas de la Matriz):** La lista de 15 palabras representa el diccionario único que el modelo construyó a partir de todo tu corpus. Nota que <code> CountVectorizer </code> aplicó automáticamente lowercasing (ej. 'Gregor' pasó a 'gregor') y eliminó los puntos finales de las oraciones. El orden de esta lista es estrictamente alfabético y dicta la posición de las columnas en la matriz inferior.
  
- **Representación Vectorial de Documentos (Filas de la Matriz):** Cada fila de la matriz corresponde exactamente a uno de tus documentos en el orden original. El modelo tradujo texto humano a vectores numéricos de dimensión $1 \times 15$ (el tamaño del vocabulario).
  
- **Análisis de Frecuencias (Fila 1 - Documento 1):** El vector [1 1 0 0 1 1 0 0 1 1 0 0 1 1 1] mapea la oración

  "Gregor Samsa despertó una mañana de un sueño tranquilo.".

  Tiene un 1 en la columna 0 ('de'), un 1 en la columna 1 ('despertó'), un 0 en la columna 2 ('el') porque esa palabra no aparece en este enunciado, y así sucesivamente.

- **Detección de Repeticiones (Fila 2 - Documento 2):** En la segunda fila, correspondiente a

  "El sueño de Gregor no era un sueño normal.",

se observa un valor de 2 en la novena posición (índice 9), que equivale a la palabra 'sueño'. El modelo capturó con precisión que el término se repite dos veces en esa oración.

- **Captura de Plurales/Géneros sin Limpieza (Fila 3 - Documento 3):** Para la oración "Una mañana tranquila y una tarde tranquila.", la matriz registra un 2 en la posición de 'tranquila' (índice 11) y un 2 en la posición de 'una' (índice 14). Sin embargo, nota que mantiene la palabra 'y' fuera porque el tokenizador de scikit-learn descarta por defecto palabras de un solo carácter, y trata a 'tranquila' y 'tranquilo' (Fila 1) como entidades completamente independientes al no haber aplicado stemming.

El mismo vectorizador se puede utilizar en otros documentos:
- Si no están incluidas en su vocabulario, entonces se ignoran. 
- Si esta incluida, la tiene en cuenta.

Los vectores codificados luego se pueden usar directamente con un algoritmo de aprendizaje automático.

<a id="section3"></a>
# <font color="#004D7F" size=6>3. Frecuencias de palabras con `TfidfVectorizer`</font>

Un problema con los recuentos simples es palabras como _the_ aparecerán muchas veces cuando no añade mucha información de contexto.

Para mitigarlo, se usa el método _Term Frequency - Inverse Document Frequency_ (TF-IDF) que significa:
- __Frecuencia de términos__: ¿Con qué frecuencia el término aparece en este documento? Mientras mayor sea la frecuencia del término en el documento, mayor será su importancia.
- __Frecuencia de documento inversa__: ¿Con qué frecuencia el término aparece en todos los documentos de la colección? Mientras mayor sea la frecuencia en los documentos, menor será la importancia del término.

`TfidfVectorizer` tokenizará documentos, aprenderá el vocabulario y el documento inverso sobre ponderaciones de frecuencia y permitirá codificar nuevos documentos.

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Más información sobre la clase [`TfidfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)

In [61]:
import string
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# ==============================================================================
# 1. PASO: ENTRADA DE DATOS (CORPUS DE PRUEBA)
# ==============================================================================
# Definimos una lista de strings que simulan nuestros documentos o renglones de base de datos.
# Nota cómo incluimos términos repetidos y variaciones de género para evaluar el comportamiento técnico.
corpus = [
    'Gregor Samsa despertó una mañana de un sueño tranquilo.',
    'El sueño de Gregor no era un sueño normal.',
    'Una mañana tranquila y una tarde tranquila.'
]

# ==============================================================================
# 2. PASO: CONFIGURACIÓN E INSTANCIACIÓN DE TFIDF_VECTORIZER
# ==============================================================================
# Inicializamos el vectorizador matemático aplicando configuraciones avanzadas:
#   - lowercase=True: Convierte todo el texto a minúsculas para unificar palabras (ej. 'El' y 'el').
#   - sublinear_tf=True: Aplica una escala logarítmica al conteo interno (1 + log(tf)). Esto evita
#     que una palabra que se repite muchas veces en un solo documento sesgue desproporcionadamente el peso.
#   - use_idf=True: Activa el cálculo de la frecuencia inversa de documento para penalizar palabras genéricas.
tfidf_model = TfidfVectorizer(
    lowercase=True, 
    sublinear_tf=True, 
    use_idf=True
)

# ==============================================================================
# 3. PASO: ENTRENAMIENTO Y AJUSTE DE LA MATRIZ (FIT_TRANSFORM)
# ==============================================================================
# .fit_transform() ejecuta dos procesos críticos en una sola operación en memoria:
#   - FIT: Analiza el corpus completo, genera el tokenizador interno por defecto y construye
#          el vocabulario único (asigna una columna fija a cada palabra en orden alfabético).
#   - TRANSFORM: Calcula los pesos TF-IDF para cada celda y devuelve una matriz dispersa de SciPy.
matriz_dispersa_tfidf = tfidf_model.fit_transform(corpus)

# Extrayemos los nombres de las columnas (los tokens únicos del vocabulario aprendido)
vocabulario_columnas = tfidf_model.get_feature_names_out()

# ==============================================================================
# 4. PASO: FORMATEO DE RESULTADOS EN UN DATAFRAME DE PANDAS
# ==============================================================================
# Convertimos la matriz dispersa a un formato denso tradicional (.toarray()) para
# poder estructurarla visualmente como una tabla de datos (filas x columnas).
df_analisis_tfidf = pd.DataFrame(
    data=matriz_dispersa_tfidf.toarray(), 
    columns=vocabulario_columnas
)

# Renombramos las filas para identificar de manera inmediata qué vector corresponde a qué documento
df_analisis_tfidf.index = [f"Documento_{i+1}" for i in range(len(corpus))]

# ==============================================================================
# 5. PASO: AUDITORÍA VISUAL Y MONITOREO DE PESOS EN COLAB
# ==============================================================================
# Desplegamos los metadatos y la tabla estructurada para comprobar las penalizaciones matemáticas:
#   - Palabras como 'de' o 'gregor' (comunes en Doc 1 y 2) tendrán pesos bajos y balanceados.
#   - Palabras como 'despertó' o 'samsa' (exclusivas del Doc 1) mantendrán pesos máximos descriptivos.
#   - La palabra 'y' del Doc 3 no figurará al ser filtrada por la expresión regular por defecto de scikit-learn (\b\w\w+\b).
print("=== COMPROBACIÓN DE VOCABULARIO (DIMENSIONES DE LA MATRIZ) ===")
print(f"Total de palabras únicas (columnas): {len(vocabulario_columnas)}")
print(list(vocabulario_columnas))
print("\n" + "="*80 + "\n")

print("=== MATRIZ DE PESOS TF-IDF DETALLADA ===")
# Redondeamos a 4 decimales únicamente para facilitar la legibilidad del reporte en la celda
df_analisis_tfidf.round(4)

=== COMPROBACIÓN DE VOCABULARIO (DIMENSIONES DE LA MATRIZ) ===
Total de palabras únicas (columnas): 15
['de', 'despertó', 'el', 'era', 'gregor', 'mañana', 'no', 'normal', 'samsa', 'sueño', 'tarde', 'tranquila', 'tranquilo', 'un', 'una']


=== MATRIZ DE PESOS TF-IDF DETALLADA ===


,de,despertó,el,era,gregor,mañana,no,normal,samsa,sueño,tarde,tranquila,tranquilo,un,una
Documento_1,0.2990,0.3931,0.0000,0.0000,0.2990,0.2990,0.0000,0.0000,0.3931,0.2990,0.0000,0.0000,0.3931,0.2990,0.2990
Documento_2,0.2797,0.0000,0.3678,0.3678,0.2797,0.0000,0.3678,0.3678,0.0000,0.4736,0.0000,0.0000,0.0000,0.2797,0.0000
Documento_3,0.0000,0.0000,0.0000,0.0000,0.0000,0.3078,0.0000,0.0000,0.0000,0.0000,0.4048,0.6854,0.0000,0.0000,0.5212


In [64]:
corpus

['Gregor Samsa despertó una mañana de un sueño tranquilo.',
 'El sueño de Gregor no era un sueño normal.',
 'Una mañana tranquila y una tarde tranquila.']

In [67]:
corpus[0]

'Gregor Samsa despertó una mañana de un sueño tranquilo.'

**ANÁLISIS DE LOS HALLAZGOS CLAVE (COMPORTAMIENTO TF-IDF)**

---

**1. El efecto de exclusividad absoluta (Doc_1)**

En el **Documento 1**, las palabras `'despertó'`, `'samsa'` y `'tranquilo'` comparten exactamente el peso más alto: **`0.3931`**.

> **Nota de Análisis:** Esto demuestra la consistencia del **IDF** (Inverse Document Frequency). Como estas tres palabras aparecen exactamente una sola vez en este documento y nunca más en todo el corpus, reciben un trato matemático idéntico, siendo marcadas como los términos más descriptivos de la primera oración.

---

In [69]:
corpus[1]

'El sueño de Gregor no era un sueño normal.'

**2. La fuerza de la repetición vs. la penalización por distribución (Doc_2)**

En el **Documento 2**, la palabra `'sueño'` alcanza un peso de **`0.4736`**, superando a términos que son exclusivos de esa oración como `'normal'` o `'era'` (`0.3678`).

> **Nota de Análisis:** Este es el comportamiento clásico de **TF-IDF**. Aunque la palabra `'sueño'` es penalizada (su IDF baja porque también se usó en el `Doc_1`), el haber aparecido **dos veces** en esta misma oración (TF) compensa la penalización del corpus y la eleva como la palabra clave indiscutible del Documento 2. Por el contrario, `'gregor'` y `'de'` caen a **`0.2797`** porque coexisten en múltiples documentos.

---

In [75]:
corpus[2]

'Una mañana tranquila y una tarde tranquila.'

**3. La supremacía de la densidad y el género (Doc_3)**

El **Documento 3** presenta los valores más altos de toda la matriz: `'tranquila'` con **`0.6854`** y `'una'` con **`0.5212`**.

> **Nota de Análisis:** Aquí vemos dos factores jugando a favor de la relevancia:
> * **Longitud del texto:** La oración es más corta, por lo que cada token representa un porcentaje más alto de la longitud total del texto (gracias a la Normalización $L_2$).
> * **Falta de preprocesamiento:** Al no haber aplicado *stemming*, el tokenizador trata a `'tranquila'` como algo completamente independiente de `'tranquilo'` (`Doc_1`). Al ser "exclusiva" de este renglón y repetirse dos veces, el modelo le asigna un peso masivo de casi `0.7`.

---

**4. La consistencia de los ceros (0.0000)**

Las celdas con valor cero demuestran la naturaleza de **matriz dispersa** (*Sparse Matrix*) del enfoque Bag of Words.

> **Nota de Análisis:** Si la palabra no existe en el documento, no aporta peso absoluto. Esto mantiene los vectores geométricamente limpios, lo cual es fundamental para optimizar la memoria y facilitar el cálculo de distancias o similitud de cosenos más adelante en el pipeline de Machine Learning.

<a id="section4"></a>
# <font color="#004D7F" size=6>4. Hashing con `HashingVectorizer`</font>

Los conteos y las frecuencias tienen la limitación de que el vocabulario se vuelve muy extenso y requerirá grandes vectores para codificar documentos que requerirán, a su vez, grandes requisitos en memoria. 

Una solución es usar un hash de palabras unidireccional para convertirlas en números enteros:
- No se requiere vocabulario
- Se puede elegir un vector de longitud fija de longitud arbitraria. 
- Inconveniente: es una función unidireccional, i.e., no hay forma de volver a convertir la codificación en una palabra.

La clase `HashingVectorizer` implementa este enfoque que se puede usar para hash palabras, luego tokenizar y codificar documentos según sea necesario. 

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Más información sobre la clase [`HashingVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.HashingVectorizer.html)

In [86]:
from sklearn.feature_extraction.text import HashingVectorizer

# 1. Definición del corpus de prueba (Mismo ejemplo en español)
corpus = [
    'Gregor Samsa despertó una mañana de un sueño tranquilo.',
    'El sueño de Gregor no era un sueño normal.',
    'Una mañana tranquila y una tarde tranquila.'
]

# 2. Instanciación del modelo HashingVectorizer
# A diferencia de Count y TF-IDF, este vectorizador no almacena un vocabulario en memoria.
# Utiliza el truco de hashing (MurmurHash3) para mapear las palabras directamente a índices.
# - n_features=2**4 (16): Limitamos el espacio de características a 16 columnas para poder visualizarlo.
#   (En producción se usan valores altos como 2**20 para evitar colisiones de tokens).
# - alternate_sign=False: Desactivamos los signos negativos para que devuelva conteos puros (Bag of Words).
# - lowercase=True: Pasa todos los tokens a minúsculas antes de aplicar la función hash.
hasher = HashingVectorizer(n_features=16, alternate_sign=False, lowercase=True)

# 3. Transformación directa del corpus
# Nota técnica: Aquí NO existe el método .fit() porque no hay un vocabulario que aprender.
# Pasamos directamente a .transform(), lo que lo hace ideal para flujos de datos en streaming (Out-of-core learning).
matriz_hashing = hasher.transform(corpus)

print("=== MATRIZ GENERADA POR HASHING VECTORIZER (16 CARACTERÍSTICAS) ===")
# Convertimos la matriz dispersa a un formato denso para inspección visual en Colab
print(matriz_hashing.toarray().round(3))

=== MATRIZ GENERADA POR HASHING VECTORIZER (16 CARACTERÍSTICAS) ===
[[0.    0.302 0.302 0.603 0.    0.    0.302 0.302 0.    0.302 0.    0.302
  0.302 0.    0.    0.   ]
 [0.    0.258 0.    0.516 0.    0.    0.258 0.    0.258 0.    0.    0.
  0.516 0.    0.516 0.   ]
 [0.    0.    0.632 0.632 0.    0.    0.    0.    0.    0.    0.316 0.316
  0.    0.    0.    0.   ]]
